Connecting to Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Basic Integration


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import random
import os
from sklearn.metrics import precision_score, recall_score
from google.colab import drive

# ✅ Mount Google Drive
drive.mount('/content/drive')

# ✅ Define Dataset Directory
DATASET_DIR = "/content/drive/MyDrive/DataSet/"

# ✅ Load the trained model
MODEL_PATH = "/content/drive/MyDrive/models/unet_Namd_trained_final.h5"
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# ✅ Function to list available test datasets (regions)
def list_regions():
    """List available test datasets formatted as '{region}_test_NDVI_NDWI.npy'."""
    regions = [f.replace("_test_NDVI_NDWI.npy", "") for f in os.listdir(DATASET_DIR) if f.endswith("_test_NDVI_NDWI.npy")]
    if not regions:
        print("❌ No test datasets found!")
        return None
    print("\n🔹 Available Regions:")
    for i, region in enumerate(regions):
        print(f"{i+1}. {region}")
    return regions

# ✅ User selects region
regions = list_regions()
if not regions:
    exit()

region_idx = int(input("Enter the number for the region: ")) - 1
region_name = regions[region_idx]
test_input_path = os.path.join(DATASET_DIR, f"{region_name}_test_NDVI_NDWI.npy")
test_label_path = os.path.join(DATASET_DIR, f"{region_name}_test_change_maps.npy")

# ✅ User selects NDVI or NDWI detection
detection_type = input("Detect NDVI (1) or NDWI (2)? Enter 1 or 2: ")
if detection_type not in ["1", "2"]:
    print("❌ Invalid selection!")
    exit()
detection_type = int(detection_type)

# ✅ Load test data
X_test = np.load(test_input_path)  # NDVI & NDWI input
Y_test = np.load(test_label_path)  # Actual Change Maps

# ✅ Normalize input data
X_test = X_test / np.max(X_test)

# ✅ Select the band for detection
band_name = "NDVI" if detection_type == 1 else "NDWI"
band_index = 0 if detection_type == 1 else 1

# ✅ Function to compute IoU, Dice Score, Precision, and Recall
def calculate_metrics(y_true, y_pred):
    """
    Calculates IoU, Dice Score, Precision, and Recall for binary segmentation maps.
    """
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    # Compute Intersection and Union
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    iou = intersection / union if union != 0 else 0

    # Compute Dice Score
    dice = (2. * intersection) / (y_true.sum() + y_pred.sum()) if (y_true.sum() + y_pred.sum()) != 0 else 0

    # Compute Precision and Recall
    precision = precision_score(y_true, y_pred, zero_division=1)
    recall = recall_score(y_true, y_pred, zero_division=1)

    return iou, dice, precision, recall

# ✅ Randomly select 5 test samples
random_indices = random.sample(range(X_test.shape[0]), min(5, X_test.shape[0]))
fig, axes = plt.subplots(len(random_indices), 3, figsize=(15, len(random_indices) * 5))

for i, idx in enumerate(random_indices):
    selected_band = X_test[idx, ..., band_index]
    actual_change = Y_test[idx].squeeze()
    predicted_change = model.predict(np.expand_dims(X_test[idx], axis=0))
    binary_prediction = (predicted_change > 0.5).astype(np.uint8).squeeze()

    # ✅ Compute IoU, Dice Score, Precision, Recall
    iou, dice, precision, recall = calculate_metrics(actual_change, binary_prediction)

    # ✅ Calculate change percentage
    change_percentage = (np.sum(binary_prediction > 0) / binary_prediction.size) * 100

    # ✅ Determine reasons for change
    if detection_type == 1:  # NDVI (Vegetation Changes)
        if change_percentage > 30:
            reason = "🔴 Deforestation or Urban Expansion"
        elif 10 < change_percentage <= 30:
            reason = "🟡 Seasonal Changes or Forest Degradation"
        else:
            reason = "🟢 Minimal Vegetation Change"
    else:  # NDWI (Water Changes)
        if change_percentage > 30:
            reason = "🔵 Flooding or River Expansion"
        elif 10 < change_percentage <= 30:
            reason = "🟠 Dry Season Effects or Water Loss"
        else:
            reason = "🟢 Stable Water Bodies"

    # ✅ Display results
    axes[i, 0].imshow(selected_band, cmap="RdYlGn" if detection_type == 1 else "Blues")
    axes[i, 0].set_title(f"{band_name} Input")

    axes[i, 1].imshow(actual_change, cmap="inferno")
    axes[i, 1].set_title("Actual Change Map")

    axes[i, 2].imshow(binary_prediction, cmap="gray")
    axes[i, 2].set_title(
        f"Predicted Change ({change_percentage:.2f}%)\n"
        f"Reason: {reason}\n"
        f"IoU: {iou:.3f} | Dice: {dice:.3f}\n"
        f"Precision: {precision:.3f} | Recall: {recall:.3f}"
    )

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()
print(f"✅ Change detection completed for {region_name} using {band_name}!")


Output hidden; open in https://colab.research.google.com to view.

**Streamlit Website Integration**


Dependency


In [3]:
!pip install rasterio
!pip install streamlit -q
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 67.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.3 MB/s eta 0:00:00


In [ ]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501

34.82.37.164
⠙

⠹⠸⠼⠴⠦⠧⠇
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.82.37.164:8501

your url is: https://swift-peaches-beg.loca.lt
2025-03-07 04:31:23.299521: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741321883.366234   11981 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741321883.383998   11981 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 04:31:23.459905: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enab

In [ ]:
y